# Genetic Algorithm Assignment
**Air Traffic Control**

30% of the overall grade for this module

Marks indciated in sections below are based on percentage of marks allocated for this module

In this assignment you must choose a problem, and attempt to use the Genetic Alogrithm that we developed in class to solve this problem.

The choice of problem has to be discussed and agreed with the lecturer.

Progression of the assignment must be demonstrated continuously throughout the semester.

Final version must be demonstrated in class before submission date.

Failure to adhere to all of the above, without express permission of the lecturer will severely impact your grade.

## The Problem (~30%)

*   Description of the problem

---

Air Traffic Control (ATC) at Dublin Airport must manage many aircraft approaching from different directions, altitudes, and speeds. Each aircraft has:

- A position (latitude, longitude, altitude)
- A movement (speed, heading, vertical rate)
- A landing status (ready, too high, too far, or both)

The goal is to slightly adjust each aircraft’s heading, speed, or vertical rate so that:

1. Aircraft stay safely apart (at least 6 km distance within 100 seconds).
2. Aircraft ready to land are guided to the runway efficiently.
3. Aircraft that are too high descend safely in time.
4. The overall penalty (conflicts, large changes, delays) is kept as low as possible.

The dataset contains 50 aircraft states captured at one moment in Dublin airspace, including position, speed, and other flight data.

---

*   Discussion of the suitablity of Genetic Algorithms

---

A Genetic Algorithm (GA) works well for this problem because:

- Large search space: Even a small number of aircraft creates many possible solutions, making simple methods impractical.
- Complex problem shape: The cost function has sudden penalties (e.g., collisions), so smooth optimisation methods don’t work well.
- Multiple goals: Safety, efficiency, and minimal changes can all be combined into one fitness score.
- Parallel evaluation: Many solutions can be tested at the same time, improving performance.

Other methods, like linear programming or greedy approaches, are less suitable because they either require simpler conditions or ignore interactions between aircraft.

---

*   Complexity of the problem

---

This is a complex problem because:

1. Many interactions: Every aircraft must be checked against others.
2. Mixed data types: Adjustments are continuous values, but constraints are strict.
3. Noisy real data: Flight data can include errors or inconsistencies.
4. Real-world rules: Safety standards and landing requirements must be followed.

# The problem and the cost function   **(~20%)**

In [35]:
import numpy as np
import pandas as pd
from copy import deepcopy

In [36]:
def closest_pass(p1, p2, v1, v2, time_window, minimum_distance):
    delta_p = p1 - p2
    delta_v = v1 - v2
    a = np.dot(delta_v, delta_v) # Speed difference between the planes
    b = 2 * np.dot(delta_p, delta_v)# Relative speed between the planes
    c = np.dot(delta_p, delta_p) # Distance between the planes

    if a == 0:
        return np.sqrt(c)
    t = -b / (2 * a)

    if t < 0:
        return 0
    if t > time_window:
        return 0    
        
    closest_distance = np.sqrt(c + t * (b + a * t))
    
    if closest_distance < minimum_distance:
        return -1000
    return 0


def convert_long_lat_alt_to_cartesian(long, lat, alt):
    y=lat * 111111
    x = long * 111111 * np.cos(np.radians(lat))
    z = alt
    return np.array([x, y, z])


def get_cartesian_velocity(vertrate, speed, heading):
    heading_rad = np.radians(heading)

    v_z = speed * np.cos(heading_rad)
    v_x = speed * np.sin(heading_rad)
    v_y = vertrate 
    
    return np.array([v_x, v_y, v_z])

## Test

In [37]:
#plane 1
lat1 =49.5354448739
long1 =2.83373033678
speed1 =181.779353382
head_1 =66.4820383263
vertrate1 = 4.55168
alt1 = 4838.7

#plane 2
lat2 =51.7063293457
long2 =-3.28788138725
speed2 = 236.560348391
head_2 =316.762391024
vertrate2 = 0.32512
alt2 = 11841.48

#plane 3
lat3 =49.542388916
long3 =2.85810771741
speed3 = 181.985246682
head_3 =246.3335256222
vertrate3 = 5.20192
alt3 = 4892.04

In [38]:
pos1 = convert_long_lat_alt_to_cartesian(long1, lat1, alt1)
vel1 = get_cartesian_velocity(vertrate1, speed1, head_1)

pos2 = convert_long_lat_alt_to_cartesian(long2, lat2, alt2)
vel2 = get_cartesian_velocity(vertrate2, speed2, head_2) 

pos3 = convert_long_lat_alt_to_cartesian(long3, lat3, alt3)
vel3 = get_cartesian_velocity(vertrate3, speed3, head_3)

In [39]:
print("Plane 1 position:", pos1, "velocity:", vel1)
print("Plane 3 position:", pos3, "velocity:", vel3)

closest_pass(pos1, pos3, vel1, vel3, 60, 6000)

Plane 1 position: [2.04336159e+05 5.50393282e+06 4.83870000e+03] velocity: [166.679856   4.55168   72.536604]
Plane 3 position: [2.06064693e+05 5.50470437e+06 4.89204000e+03] velocity: [-166.679856    5.20192   -73.051048]


-1000

## Import Dataset

In [40]:
flights_df = pd.read_csv("Data/50_flights.csv")
print(flights_df.head())

         time  icao24        lat       lon    velocity     heading  vertrate  \
0  1527548460  4ca1d2  49.568544  2.950126  183.341375   66.168099   4.87680   
1  1527548460  4ca5ea  51.783127 -3.404541  236.185829  316.676901   0.32512   
2  1527548470  4ca5ea  51.798535 -3.427963  236.185829  316.676901   0.00000   
3  1527548470  4ca1d2  49.576538  2.972380  186.408412   54.774808   6.17728   
4  1527548470  4ca8d7  53.678501 -5.458186  158.452093  269.627955  -5.20192   

   callsign  onground  alert    spi  squawk  baroaltitude  geoaltitude  \
0     RYR5      False  False  False   674.0       4953.00      5097.78   
1  RYR17XT      False  False  False  5701.0      11582.40     11833.86   
2  RYR17XT      False  False  False  5701.0      11582.40     11833.86   
3     52GH      False  False  False   674.0       4998.72      5151.12   
4  RYR73UF      False  False  False  2522.0       3512.82      3680.46   

   lastposupdate   lastcontact  
0   1.527548e+09  1.527548e+09  
1   1.52

## Load flight data

In [41]:
positions = []
velocities = []

for idx, row in flights_df.iterrows():
    pos = convert_long_lat_alt_to_cartesian(row["lon"], row["lat"], row["geoaltitude"])
    vel = get_cartesian_velocity(row["vertrate"], row["velocity"], row["heading"])
    positions.append(pos)
    velocities.append(vel)

flights_df["converted_position"] = positions
flights_df["cartesian_velocity"] = velocities

print(flights_df.head())

         time  icao24        lat       lon    velocity     heading  vertrate  \
0  1527548460  4ca1d2  49.568544  2.950126  183.341375   66.168099   4.87680   
1  1527548460  4ca5ea  51.783127 -3.404541  236.185829  316.676901   0.32512   
2  1527548470  4ca5ea  51.798535 -3.427963  236.185829  316.676901   0.00000   
3  1527548470  4ca1d2  49.576538  2.972380  186.408412   54.774808   6.17728   
4  1527548470  4ca8d7  53.678501 -5.458186  158.452093  269.627955  -5.20192   

   callsign  onground  alert    spi  squawk  baroaltitude  geoaltitude  \
0     RYR5      False  False  False   674.0       4953.00      5097.78   
1  RYR17XT      False  False  False  5701.0      11582.40     11833.86   
2  RYR17XT      False  False  False  5701.0      11582.40     11833.86   
3     52GH      False  False  False   674.0       4998.72      5151.12   
4  RYR73UF      False  False  False  2522.0       3512.82      3680.46   

   lastposupdate   lastcontact  \
0   1.527548e+09  1.527548e+09   
1   1.

## Check for conflics

In [42]:
TIME_WINDOW = 100      
MIN_DISTANCE = 6000   

conflict_pairs = []
n = len(flights_df)

for i in range(n):
    for j in range(i + 1, n):
        p1 = flights_df.iloc[i]["converted_position"]
        p2 = flights_df.iloc[j]["converted_position"]
        v1 = flights_df.iloc[i]["cartesian_velocity"]
        v2 = flights_df.iloc[j]["cartesian_velocity"]

        score = closest_pass(p1, p2, v1, v2, TIME_WINDOW, MIN_DISTANCE)

        if score < 0:
            conflict_pairs.append({
                "plane_A": flights_df.iloc[i]["callsign"].strip(),
                "plane_B": flights_df.iloc[j]["callsign"].strip(),
                "conflict_score": score
            })

conflicts_df = pd.DataFrame(conflict_pairs)

if conflicts_df.empty:
    print("No conflicts detected within the time window.")
else:
    print(f"Conflicts: {len(conflicts_df)}")
    print(conflicts_df.to_string(index=False))


Conflicts: 7
plane_A plane_B  conflict_score
   RYR5    52GH           -1000
   RYR5   RY2GH           -1000
   52GH   RY2GH           -1000
RYR66CU RYR66CU           -1000
EIN76HJ EIN76HJ           -1000
 RYR663  RYR663           -1000
RYR73UF RYR73UF           -1000


#### Find all planes' distance to the Dublins' runways

In [43]:
runways_df = pd.read_csv("Data/dublin_runways.csv")
print(runways_df.head())

       id  airport_ref airport_ident  length_ft  width_ft surface  lighted  \
0  356512         2533          EIDW    10203.0     148.0     CON        1   
1  235634         2533          EIDW     8652.0     148.0    ASPH        1   
2  235636         2533          EIDW     6798.0     148.0     ASP        1   

   closed le_ident  le_latitude_deg  le_longitude_deg  le_elevation_ft  \
0       0      10L         53.43716          -6.28062            235.0   
1       0      10R         53.42243          -6.29007            242.0   
2       0       16         53.43700          -6.26198            217.0   

   le_heading_degT  le_displaced_threshold_ft he_ident  he_latitude_deg  \
0             97.0                        NaN      28R        53.435200   
1             95.0                        NaN      28L        53.420300   
2            157.0                        NaN       34        53.419899   

   he_longitude_deg  he_elevation_ft  he_heading_degT  \
0          -6.24496            2

#### Convert runway threshold positions to Cartesian

Runways: low end and high end

1. Every runway has two ends, you can land from either direction. The two ends are named by their magnetic heading (rounded to the nearest 10°, divided by 10).

- Low end (le_) — the end with the lower runway number (e.g., Runway 09, facing ~90°/East)

- High end (he_) — the end with the higher runway number (e.g., Runway 27, facing ~270°/West)

So a runway named 09/27 has:

le_ident = "09", le_heading ≈ 90°, le_pos = GPS position of the east threshold

he_ident = "27", he_heading ≈ 270°, he_pos = GPS position of the west threshold

2. The letters indicate parallel runways at airports with multiple runways facing the same direction:

- L — Left
- R — Right
- C — Center (when there are 3 parallel runways)

So 10L/28R and 10R/28L are two side-by-side runways both oriented ~100°/280°. From the pilot's perspective on approach, one is to their left and one to their right.

The letter is always consistent across both ends. If the low end is 10L, the high end is 28R (not 28L).

In [44]:
runway_thresholds = []

for idx, row in runways_df.iterrows():
    low_end_pos = convert_long_lat_alt_to_cartesian(row["le_longitude_deg"], row["le_latitude_deg"], 0)
    high_end_pos = convert_long_lat_alt_to_cartesian(row["he_longitude_deg"], row["he_latitude_deg"], 0)
    runway_thresholds.append({
        "runway_name": str(row["le_ident"]) + "/" + str(row["he_ident"]),
        "le_ident": row["le_ident"],
        "he_ident": row["he_ident"],
        "le_pos": low_end_pos,
        "he_pos": high_end_pos,
        "le_heading": row["le_heading_degT"],
        "he_heading": row["he_heading_degT"],
        "length_ft": row["length_ft"]
    })

print(runway_thresholds)

[{'runway_name': '10L/28R', 'le_ident': '10L', 'he_ident': '28R', 'le_pos': array([-415709.68442539, 5937456.28476   ,       0.        ]), 'he_pos': array([-413368.4400408, 5937238.5072   ,       0.       ]), 'le_heading': 97.0, 'he_heading': 278.0, 'length_ft': 10203.0}, {'runway_name': '10R/28L', 'le_ident': '10R', 'he_ident': '28L', 'le_pos': array([-416479.47665226, 5935819.61973   ,       0.        ]), 'he_pos': array([-413885.48966363, 5935582.9533    ,       0.        ]), 'le_heading': 95.0, 'he_heading': 275.0, 'length_ft': 8652.0}, {'runway_name': '16/34', 'le_ident': '16', 'he_ident': '34', 'le_pos': array([-414477.47693878, 5937438.507     ,       0.        ]), 'he_pos': array([-413823.83894118, 5935538.397789  ,       0.        ]), 'le_heading': 157.0, 'he_heading': 337.0, 'length_ft': 6798.0}]


#### Distance to Runway and Landing Readiness Check

ICAO Standard: 3 degree glide path

Final approach intercept: between 3 NM and 10 NM from threshold

Rule of thumb for Top of Descent (TOD): 3 nautical miles per 1,000 feet of altitude

1 nautical mile = 1852 meters

Calculate the horizontal distance (in meters) from a plane to a runway threshold.

In [45]:
NM_TO_METERS = 1852
MAX_APPROACH_DISTANCE_NM = 10 # 10 nautical miles from threshold
MAX_APPROACH_ALTITUDE_FT = 3000 # 3000 ft above runway elevation
DUBLIN_AIRPORT_ELEVATION_FT = 242

MAX_APPROACH_DISTANCE_M = MAX_APPROACH_DISTANCE_NM * NM_TO_METERS # 18,520 m
MAX_APPROACH_ALT_TOTAL = DUBLIN_AIRPORT_ELEVATION_FT + MAX_APPROACH_ALTITUDE_FT # 3242 ft

def get_distance_to_runway(plane_pos, runway_pos):
    dx = plane_pos[0] - runway_pos[0]
    dy = plane_pos[1] - runway_pos[1]
    
    horizontal_distance = np.sqrt(dx * dx + dy * dy)
    
    return horizontal_distance

Calculate the distance at which a plane should begin its approach.

ICAO standard: 3 NM per 1,000 feet of altitude (3 degree glide path).

In [46]:
def get_landing_start_distance(plane_alt_ft):
    altitude_thousands = plane_alt_ft / 1000.0
    distance_nm = altitude_thousands * 3.0
    distance_meters = distance_nm * NM_TO_METERS
    
    return distance_meters

Find the closest runway to a plane, considering:
1. Horizontal distance to each runway threshold
2. Which end of the runway to approach based on plane heading

Returns the closest runway info and whether the plane is within landing range.

Distance Check (MAX_APPROACH_DISTANCE_M): "The plane must be within 10 NM (18.5 km) of a runway threshold. The glideslope intercept usually occurs between 5 to 6 nautical miles from the runway threshold FlightInsight, and ATC will use an approach gate at least 5 NM from the landing threshold."

Altitude Check (MAX_APPROACH_ALT_TOTAL): "The plane must be below 3,000 ft above airport elevation (3,242 ft above sea level for Dublin at 242 ft). The intermediate approach altitude generally intercepts the glide path at heights from 1,000 ft to 3,000 ft above runway elevation."

In [47]:
def find_closest_runway(plane_lat, plane_lon, plane_alt, plane_heading):

    plane_pos = convert_long_lat_alt_to_cartesian(plane_lon, plane_lat, plane_alt)
    
    best_runway_name = ""
    best_approach_end = ""
    best_distance = 0
    best_heading_diff = 0

    for rw in runway_thresholds:
        dist_low_end = get_distance_to_runway(plane_pos, rw["le_pos"])
        dist_high_end = get_distance_to_runway(plane_pos, rw["he_pos"])

        # Check heading alignment with lown end approach
        # When landing on low end, plane flies heading = le_heading
        diff_le = abs(plane_heading - rw["le_heading"])
        if diff_le > 180:
            diff_le = 360 - diff_le

        # Check heading alignment with high end approach
        # When landing on high end, plane flies heading = he_heading
        diff_he = abs(plane_heading - rw["he_heading"])
        if diff_he > 180:
            diff_he = 360 - diff_he

        # Pick the end that best matches the plane heading
        if diff_le < diff_he:
            approach_dist = dist_low_end
            approach_end = rw["le_ident"]
            heading_diff = diff_le
        else:
            approach_dist = dist_high_end
            approach_end = rw["he_ident"]
            heading_diff = diff_he

        # Pick the runway with the shortest distance
        if approach_dist < best_distance:
            best_distance = approach_dist
            best_runway_name = rw["runway_name"]
            best_approach_end = approach_end
            best_heading_diff = heading_diff

        distance_ok = False
        altitude_ok = False
 
        if best_distance <= MAX_APPROACH_DISTANCE_M:
            distance_ok = True
    
        if plane_alt <= MAX_APPROACH_ALT_TOTAL:
            altitude_ok = True

        is_ready = False
        status = ""

        if distance_ok and altitude_ok:
            is_ready = True
            status = "READY TO LAND"
        elif distance_ok and not altitude_ok:
            is_ready = False
            status = "TOO HIGH"
        elif not distance_ok and altitude_ok:
            is_ready = False
            status = "TOO FAR"
        else:
            is_ready = False
            status = "TOO FAR & TOO HIGH"

        return best_runway_name, best_approach_end, best_distance, best_heading_diff, distance_ok, altitude_ok, is_ready, status

Check all planes

In [48]:
ready_count = 0
too_high_count = 0
too_far_count = 0
too_far_high_count = 0

for idx, row in flights_df.iterrows():
    callsign = row["callsign"].strip()
    lat = row["lat"]
    lon = row["lon"]
    alt = row["geoaltitude"]
    heading = row["heading"]
    
    rw_name, appr_end, dist, hdg_diff, dist_ok, alt_ok, is_ready, status = find_closest_runway(lat, lon, alt, heading)

    if is_ready:
        ready_count = ready_count + 1
    elif status == "TOO HIGH":
        too_high_count = too_high_count + 1
    elif status == "TOO FAR":
        too_far_count = too_far_count + 1
    else:
        too_far_high_count = too_far_high_count + 1
 
total = ready_count + too_high_count + too_far_count + too_far_high_count

print(f"Ready to land: {ready_count}")
print(f"Too high: {too_high_count}")
print(f"Too far: {too_far_count}")
print(f"Too far and too high: {too_far_high_count}")
print(f"Total planes: {total}")

Ready to land: 6
Too high: 41
Too far: 0
Too far and too high: 0
Total planes: 47


#### Check if planes are approaching or passing Dublin Airport

1. Get the plane's current position
2. Get where the plane will be after a short time (60 seconds)
3. If the plane is closer to the airport after 60 seconds, it is approaching
4. If the plane is further from the airport after 60 seconds, it is passing

This tells which planes are heading towards Dublin and which ones are just flying over on their way somewhere else.

In [49]:
DUBLIN_AIRPORT_LAT = 53.4287
DUBLIN_AIRPORT_LON = -6.2621
DUBLIN_CENTER_POS = convert_long_lat_alt_to_cartesian(DUBLIN_AIRPORT_LON, DUBLIN_AIRPORT_LAT, 0)
TIME_STEP = 60

print(DUBLIN_CENTER_POS)

[-414566.37290465 5936516.2857           0.        ]


In [50]:
def check_approaching_or_passing(plane_pos, plane_vel):
    dx_now = plane_pos[0] - DUBLIN_CENTER_POS[0]
    dy_now = plane_pos[1] - DUBLIN_CENTER_POS[1]
    dist_now = np.sqrt(dx_now * dx_now + dy_now * dy_now)
    
    # Future position after 60 seconds
    future_x = plane_pos[0] + plane_vel[0] * TIME_STEP
    future_y = plane_pos[1] + plane_vel[1] * TIME_STEP
    
    # Future horizontal distance to Dublin
    dx_future = future_x - DUBLIN_CENTER_POS[0]
    dy_future = future_y - DUBLIN_CENTER_POS[1]
    dist_future = np.sqrt(dx_future * dx_future + dy_future * dy_future)
    
    if dist_future < dist_now:
        is_approaching = True
        direction = "APPROACHING"
    else:
        is_approaching = False
        direction = "PASSING"
    
    return dist_now, dist_future, is_approaching, direction
 

Check all planes

In [51]:
approaching_count = 0
passing_count = 0
approaching_planes = []
passing_planes = []
 
for idx, row in flights_df.iterrows():
    callsign = row["callsign"].strip()
    plane_pos = row["converted_position"]
    plane_vel = row["cartesian_velocity"]
    
    dist_now, dist_future, is_approaching, direction = check_approaching_or_passing(plane_pos, plane_vel)
    
    dist_now_km = dist_now / 1000.0
    dist_future_km = dist_future / 1000.0
    change_km = (dist_future - dist_now) / 1000.0
    
    print(f"\nPlane: {callsign}\nCurrent distance: {dist_now_km} km, Future distance: {dist_future_km} km, Change: {change_km} km, Status: {direction}")
    
    if is_approaching:
        approaching_count = approaching_count + 1
        approaching_planes.append(idx)
    else:
        passing_count = passing_count + 1
        passing_planes.append(idx)
 
print(f"Approaching Dublin: {approaching_count}")
print(f"Passing Dublin: {passing_count}")
print(f"Total: {approaching_count + passing_count}")


Plane: RYR5
Current distance: 759.7889825694629 km, Future distance: 767.9525320167133 km, Change: 8.163549447250553 km, Status: PASSING

Plane: RYR17XT
Current distance: 256.9588254731795 km, Future distance: 250.20860315597054 km, Change: -6.7502223172089435 km, Status: APPROACHING

Plane: RYR17XT
Current distance: 254.66593169006165 km, Future distance: 247.9276499845436 km, Change: -6.738281705518078 km, Status: APPROACHING

Plane: 52GH
Current distance: 760.5839754053707 km, Future distance: 767.9472333189672 km, Change: 7.363257913596462 km, Status: PASSING

Plane: RYR73UF
Current distance: 61.91753975098236 km, Future distance: 53.42803877530887 km, Change: -8.489500975673486 km, Status: APPROACHING

Plane: RYR39BN
Current distance: 38.859932235554076 km, Future distance: 37.55202780227096 km, Change: -1.3079044332831182 km, Status: APPROACHING

Plane: EIN54A
Current distance: 577.8427171871933 km, Future distance: 569.5086464734599 km, Change: -8.334070713733439 km, Status: AP

# Cost function

The cost function takes a chromosome (a list of adjusted values for each plane) and returns a score. A lower score is better. The score is made up of two parts:

1. A big penalty for every pair of planes that would conflict (using closest_pass)
2. A small penalty for how much each plane was changed from its original values - we want the smallest possible changes that still fix the conflicts

In [52]:
approaching_df = flights_df.iloc[approaching_planes].reset_index(drop=True)
print("Number of approaching planes:", len(approaching_df))
approaching_df.head()

Number of approaching planes: 44


,time,icao24,lat,lon,velocity,heading,vertrate,callsign,onground,alert,spi,squawk,baroaltitude,geoaltitude,lastposupdate,lastcontact,converted_position,cartesian_velocity
0,1527548460,4ca5ea,51.783127,-3.404541,236.185829,316.676901,0.32512,RYR17XT,False,False,False,5701.0,11582.40,11833.86,1.527548e+09,1.527548e+09,"[-234020.2744421986, 5753674.976585937, 11833.86]","[-162.04985999911915, 0.32512, 171.82429600059..."
1,1527548470,4ca5ea,51.798535,-3.427963,236.185829,316.676901,0.00000,RYR17XT,False,False,False,5701.0,11582.40,11833.86,1.527548e+09,1.527548e+09,"[-235549.77772521673, 5755387.062729404, 11833...","[-162.04985999911915, 0.0, 171.82429600059547]"
2,1527548470,4ca8d7,53.678501,-5.458186,158.452093,269.627955,-5.20192,RYR73UF,False,False,False,2522.0,3512.82,3680.46,1.527548e+09,1.527548e+09,"[-359218.32291968726, 5964271.917411007, 3680.46]","[-158.4487519997624, -5.20192, -1.028887999436..."
3,1527548470,4ca303,53.768784,-6.174630,124.190021,261.903463,-4.55168,RYR39BN,False,False,False,2002.0,11582.40,2011.68,1.527548e+09,1.527548e+09,"[-405498.01916235033, 5974303.311168492, 2011.68]","[-122.95211599950525, -4.55168, -17.4910960001..."
4,1527548470,4ca295,49.342160,-0.789715,231.440917,277.535732,-1.62560,EIN54A,False,False,False,6761.0,11574.78,11826.24,1.527548e+09,1.527548e+09,"[-57170.04838345661, 5482456.737815558, 11826.24]","[-229.44202400011028, -1.6256, 30.35219600137182]"


Small penalty for plane values change is for forcing grenetic algorithm to prefer solutions that fix conflicts with the smallest possible changes to the planes' current trajectories. So it will only change a plane's course if it actually needs to, not just randomly. 

Example: the Genetic algorithm might find solutions like "turn every plane 29 degrees and change every speed by 28 m/s", technically no conflicts, but completely unrealistic and dangerous in real ATC

In [53]:
def cost_function(chromosome):
    total_cost = 0
    new_velocities = []

    for i in range(len(approaching_df)):
        heading = chromosome[i * 3 + 0]
        speed = chromosome[i * 3 + 1]
        vertrate = chromosome[i * 3 + 2]

        new_heading = approaching_df.iloc[i]["heading"] + heading
        new_speed = approaching_df.iloc[i]["velocity"] + speed
        new_vertrate = approaching_df.iloc[i]["vertrate"] + vertrate

        new_vel = get_cartesian_velocity(new_vertrate, new_speed, new_heading)
        new_velocities.append(new_vel)

        # Small penalty for changing values from original
        total_cost = total_cost + abs(heading)
        total_cost = total_cost + abs(speed)
        total_cost = total_cost + abs(vertrate)

    # Big penalty for every conflict pair
    for i in range(len(approaching_df)):
        for j in range(i + 1, len(approaching_df)):
            p1 = approaching_df.iloc[i]["converted_position"]
            p2 = approaching_df.iloc[j]["converted_position"]
            v1 = new_velocities[i]
            v2 = new_velocities[j]

            score = closest_pass(p1, p2, v1, v2, TIME_WINDOW, MIN_DISTANCE)

            if score < 0:
                total_cost = total_cost + 1000

    return total_cost

# The Individual **(~30%)**


*   Chromosone
*   Crossover
*   Mutation

## Discussion and Justification on the Approaches Taken

---

### Chromosome Design

Each individual represents a full set of ATC instructions for all aircraft.

The chromosome is a simple array with 3 values per plane:

- Heading change (delta_heading): turn left/right (−30° to +30°)
- Speed change (delta_speed): speed up or slow down (−30 to +30 m/s)
- Vertical rate change (delta_vertRate): adjust climb/descent (−5 to +5 m/s)

So for N planes, the chromosome has 3 × N values.

**Why delta and not absolute values?**

Instead of setting completely new values, we only adjust the current ones.
This is better because:

1. Doing nothing (all zeros) is a valid starting point
2. The algorithm only needs to find small improvements
3. It avoids wasting time rediscovering current flight conditions

**Why these three genes per plane?**

They control movement in all directions:

- Heading: left/right separation
- Speed: spacing between planes on the same path
- Vertical rate: altitude separation

Together, they fully control aircraft positioning.

**Example:**

If a plane has values [+15, -10, -2], it means:

Turn 15° right
Slow down by 10 m/s
Descend faster by 2 m/s

### Crossover

Same as the class example.

### Mutation

Same as the class example.

---

In [ ]:
class Individual:
    def __init__(self):
        self.chromosome = []

        for i in range(len(approaching_df)):
            delta_heading = np.random.uniform(-30, 30)
            delta_speed = np.random.uniform(-30, 30)
            delta_vertrate = np.random.uniform(-5, 5)

            self.chromosome.append(delta_heading)
            self.chromosome.append(delta_speed)
            self.chromosome.append(delta_vertrate)

        self.chromosome = np.array(self.chromosome)
        self.cost = cost_function(self.chromosome)

    def Crossover(self, other, epsilon):
        child1 = deepcopy(self)
        child2 = deepcopy(other)

        alpha = np.random.uniform(-epsilon, 1 + epsilon)

        child1.chromosome = alpha * self.chromosome + (1 - alpha) * other.chromosome
        child2.chromosome = (1 - alpha) * self.chromosome + alpha * other.chromosome

        return child1, child2

    def Mutate(self, mutation_rate, mutation_range):
        for i in range(len(self.chromosome)):
            if np.random.uniform(0, 1) < mutation_rate:
                self.chromosome[i] = self.chromosome[i] + np.random.randn() * mutation_range

## Running the algorithm  **(~10%)**

*   Parameter choices
*   Modifications (if any) to run_genetic
*   Rationale for the above


---
### Parameter

- population: 100 individuals. Enough to have diversity without being too slow.
- generations: 100. The cost usually stops improving after about 60-70 generations.
- crossover_exploration: 0.1 same as class example.
- mutation_rate: 0.2 same as class example.
- mutation_range: 1.0 small changes per mutation since the genes are already small delta values.
- child_factor: 1 so the number of children equals the population size each generation.

---

In [58]:
class Parameters:
    def __init__(self):
        self.population = 100
        self.maximum_number_of_generations = 100
        self.crossover_exploration = 0.1
        self.mutation_rate = 0.2
        self.mutation_range = 1.0
        self.child_factor = 1

In [59]:
def get_parents(population):
    index1 = np.random.randint(0, len(population))
    index2 = np.random.randint(0, len(population))
    if index1 == index2:
        return get_parents(population)
    parent1 = population[index1]
    parent2 = population[index2]
    return parent1, parent2


def run_genetic(params):
    number_in_population = params.population
    maximum_number_of_generations = params.maximum_number_of_generations
    crossover_exploration = params.crossover_exploration
    mutation_rate = params.mutation_rate
    mutation_range = params.mutation_range
    children_per_generation = number_in_population * params.child_factor

    # Generate the initial population
    population = []
    best_solution = Individual()
    best_cost = np.inf

    for i in range(number_in_population):
        new_individual = Individual()
        if new_individual.cost < best_cost:
            best_cost = new_individual.cost
            best_solution = deepcopy(new_individual)
        population.append(new_individual)

    # Iterate for each generation
    for generation in range(maximum_number_of_generations):
        children = []
        while len(children) < children_per_generation:
            parent1, parent2 = get_parents(population)
            child1, child2 = parent1.Crossover(parent2, crossover_exploration)
            child1.Mutate(mutation_rate, mutation_range)
            child2.Mutate(mutation_rate, mutation_range)
            child1.cost = cost_function(child1.chromosome)
            child2.cost = cost_function(child2.chromosome)
            children.append(child1)
            children.append(child2)

        population = population + children
        population = sorted(population, key=lambda x: x.cost)
        population = population[:number_in_population]

        if population[0].cost < best_solution.cost:
            best_solution = deepcopy(population[0])
            best_cost = best_solution.cost

        print(best_cost)

    return best_solution

In [60]:
params = Parameters()
best = run_genetic(params)

9434.881664881725
9434.881664881725
9434.881664881725
8680.412828641394
8626.958339481072
6536.1478575450565
6536.1478575450565
6536.1478575450565
5582.149412960315
5582.149412960315
5573.154897564355
5507.226106896205
5507.226106896205
5507.226106896205
5507.226106896205
5507.226106896205
5507.226106896205
5507.226106896205
5507.226106896205
5507.226106896205
5507.226106896205
5507.226106896205
5507.226106896205
5507.226106896205
5507.226106896205
5506.822624072852
5504.31136540748
5499.299686827784
5499.299686827784
5499.299686827784
5497.001436668155
5494.834265498208
5494.2402058588395
5488.34184680447
5488.34184680447
5484.86349201122
5482.569724731873
5477.935399224223
5476.301873433802
5476.301873433802
5476.301873433802
5470.721448041231
5468.098963115173
5466.338800453601
5464.458865626162
5463.206830655377
5456.1485441929835
5454.298222447313
5452.939154099626
5449.31051951757
5449.31051951757
5442.7421382919065
5437.771735039877
5437.771735039877
5437.771735039877
5434.87487

In [61]:
print("Best cost:", best.cost)

Best cost: 3401.2904651211165


In [ ]:
for i in range(len(approaching_df)):
    callsign = approaching_df.iloc[i]["callsign"].strip()
    delta_heading = best.chromosome[i * 3 + 0]
    delta_speed = best.chromosome[i * 3 + 1]
    delta_vertrate = best.chromosome[i * 3 + 2]
    print(callsign, "  heading change:", round(delta_heading, 2), "  speed change:", round(delta_speed, 2), "  vertrate change:", round(delta_vertrate, 2))

RYR17XT   heading change: 5.31   speed change: -5.09   vertrate change: -0.24
RYR17XT   heading change: -4.73   speed change: -2.59   vertrate change: -0.21
RYR73UF   heading change: -6.22   speed change: -5.95   vertrate change: -0.85
RYR39BN   heading change: -2.26   speed change: -6.19   vertrate change: -0.48
EIN54A   heading change: 0.95   speed change: -7.48   vertrate change: 0.79
RYR74UE   heading change: 0.04   speed change: -2.08   vertrate change: 1.59
EIN56V   heading change: 9.43   speed change: 2.37   vertrate change: 0.54
RYR663   heading change: 8.13   speed change: -1.05   vertrate change: 1.26
RYR66CU   heading change: 0.96   speed change: -0.08   vertrate change: -0.47
EIN4JR   heading change: -2.83   speed change: -9.19   vertrate change: -0.93
RYR87FC   heading change: 9.95   speed change: 0.48   vertrate change: 0.26
EIN76HJ   heading change: -2.06   speed change: -0.04   vertrate change: -0.05
EIN7RE   heading change: -3.0   speed change: 8.21   vertrate change: 

## Results and conclusions    **(~10%)**